In [6]:
# Install packages (Google Colab)
!pip install -q gradio transformers torch black autopep8 reportlab

import re
import ast
import gradio as gr
import autopep8
import black
from transformers import pipeline
from reportlab.pdfgen import canvas

# -----------------------------
# Chatbot Configuration
# -----------------------------
class ChatbotConfig:
    MODEL_NAME = "gpt2"   # changed to small model
    DEFAULT_TEMP = 0.7
    DEFAULT_MAX_TOKENS = 100
    DEFAULT_TOP_P = 0.95


# -----------------------------
# Load Model
# -----------------------------
model = pipeline(
    "text-generation",
    model=ChatbotConfig.MODEL_NAME
)

conversation_history = []

# -----------------------------
# Detect Python Code
# -----------------------------
def detect_python_code(text):

    patterns = [
        r'def\s+\w+\s*\(',
        r'class\s+\w+',
        r'import\s+\w+',
        r'from\s+\w+\s+import',
        r'if\s+.*:',
        r'for\s+.*\s+in\s+',
        r'while\s+.*:'
    ]

    for p in patterns:
        if re.search(p, text):
            return True

    return False


# -----------------------------
# Check Python Syntax
# -----------------------------
def check_syntax(code):

    try:
        ast.parse(code)
        return True, "No syntax errors"

    except SyntaxError as e:
        return False, f"Syntax Error line {e.lineno}: {e.msg}"


# -----------------------------
# Fix Python Code
# -----------------------------
def fix_python_code(code):

    try:
        fixed = autopep8.fix_code(code)

        try:
            fixed = black.format_str(fixed, mode=black.FileMode())
        except:
            pass

        return fixed

    except:
        return code


# -----------------------------
# Generate AI Response
# -----------------------------
def generate_response(message, temperature, max_tokens, top_p):

    output = model(
        message,
        max_new_tokens=int(max_tokens),
        temperature=temperature,
        do_sample=True,
        top_p=top_p
    )

    reply = output[0]["generated_text"]

    return reply


# -----------------------------
# Main Chat Function
# -----------------------------
def chatbot(message, history, temperature, max_tokens, top_p):

    if detect_python_code(message):

        ok, err = check_syntax(message)

        if not ok:
            fixed = fix_python_code(message)
            history.append((message, f"Fixed Code:\n{fixed}\n\nError: {err}"))
            return history

    reply = generate_response(message, temperature, max_tokens, top_p)

    history.append((message, reply))

    return history


# -----------------------------
# Gradio Interface
# -----------------------------
with gr.Blocks() as demo:

    gr.Markdown("# Syntax Surgeon - AI Code Fixer")

    chatbot_ui = gr.Chatbot(height=400)

    msg = gr.Textbox(label="Enter Message")

    send = gr.Button("Send")

    temp = gr.Slider(0.1,1,value=0.7,label="Temperature")
    tokens = gr.Slider(50,200,value=100,label="Max Tokens")
    top_p = gr.Slider(0.1,1,value=0.95,label="Top-P")

    send.click(
        chatbot,
        inputs=[msg, chatbot_ui, temp, tokens, top_p],
        outputs=chatbot_ui
    )

demo.launch(share=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.4/91.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 11.7 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/tmp/ipykernel_450/3538385618.py:132: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot_ui = gr.Chatbot(height=400)
/tmp/ipykernel_450/3538385618.py:132: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot_ui = gr.Chatbot(height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ff970b5963058d29fe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
